# PCOS Quantum-Kernel Models in PennyLane

PennyLane replacement for the NumPy simulator. Run `pip install pennylane scikit-learn pandas numpy` first. The final cell reads `run.py` (script version of `run.ipynb`) for data prep and runs a self-check against the simulator.

In [ ]:
"""
PCOS comorbidity risk - PennyLane quantum-kernel models.
Drop-in replacement for the NumPy simulator used in run.py/tune.py.
NOT executed in the authoring sandbox (no network / no PennyLane) - run the self-check below first.

    pip install pennylane scikit-learn pandas numpy
    python pcos_pennylane_qml.py
Requires run.py's data-prep + augmentation code (exec'd below) in the same folder.
"""
import numpy as np, pennylane as qml
from sklearn.svm import SVC

def make_kernel(nq, reps=2):
    dev = qml.device("default.qubit", wires=nq)
    pairs = [(i, j) for i in range(nq) for j in range(i + 1, nq)]

    def fmap(x):                       # Havlicek ZZ feature map
        for _ in range(reps):
            for w in range(nq): qml.Hadamard(wires=w)
            for w in range(nq): qml.RZ(-2 * x[w], wires=w)               # exp(+i x Z)
            for i, j in pairs:
                qml.IsingZZ(-2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])
    # NOTE: reference sim applies phase THEN H per rep starting from H|0>; reorder identically:
    def fmap_ref(x):
        for w in range(nq): qml.Hadamard(wires=w)
        for _ in range(reps):
            for w in range(nq): qml.RZ(-2 * x[w], wires=w)
            for i, j in pairs: qml.IsingZZ(-2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])
            for w in range(nq): qml.Hadamard(wires=w)

    @qml.qnode(dev)
    def circ(x1, x2):                  # fidelity = |<0| U(x2)^dag U(x1) |0>|^2
        fmap_ref(x1); qml.adjoint(fmap_ref)(x2)
        return qml.probs(wires=range(nq))
    return lambda a, b: circ(a, b)[0]

def kmat(A, B, k):
    return np.array([[k(a, b) for b in B] for a in A])

In [ ]:
class PLQSVC:
    """Quantum-kernel SVM; mirrors QSVC/QKernel in run.py (PCA/MI -> [0,pi]*gamma -> ZZ kernel -> SVC)."""
    def __init__(self, nq=4, reps=2, reduce="mi", gamma=0.2, C=4): self.nq, self.reps, self.reduce, self.gamma, self.C = nq, reps, reduce, gamma, C
    def fit(self, X, y):
        from sklearn.preprocessing import StandardScaler, MinMaxScaler
        from sklearn.decomposition import PCA
        from sklearn.feature_selection import SelectKBest, mutual_info_classif
        self.sc = StandardScaler().fit(X); Z = self.sc.transform(X)
        self.red = PCA(self.nq, random_state=0).fit(Z) if self.reduce == "pca" else SelectKBest(mutual_info_classif, k=self.nq).fit(Z, y)
        self.mm = MinMaxScaler((0, np.pi)).fit(self.red.transform(Z))
        self.Q = self._q(X); self.k = make_kernel(self.nq, self.reps)
        self.m = SVC(kernel="precomputed", C=self.C, class_weight="balanced", probability=True, random_state=0).fit(kmat(self.Q, self.Q, self.k), y)
        return self
    def _q(self, X): return np.clip(self.mm.transform(self.red.transform(self.sc.transform(X))), 0, np.pi) * self.gamma
    def predict_proba(self, X): return self.m.predict_proba(kmat(self._q(X), self.Q, self.k))

In [ ]:
if __name__ == "__main__":
    # 1) self-check: PennyLane kernel must equal the NumPy simulator used for the reported results
    src = open("run.py").read().split("out = {")[0]; exec(src)       # defines tasks, zz_map, augment, ...
    X, y, _ = tasks["Depression/anxiety risk"]
    q = PLQSVC(nq=4, gamma=0.2).fit(X[:40], y[:40])
    ref = zz_map(q.Q[:5], 2, "full"); Kref = np.abs(ref.conj() @ ref.T) ** 2
    Kpl = kmat(q.Q[:5], q.Q[:5], q.k)
    print("max |PennyLane - NumPy sim| =", np.abs(Kpl - Kref).max(), "(expect ~1e-8)")
    # 2) CV with augmentation, same protocol as results
    MODELS["PennyLane QSVC (MI-4q)"] = lambda: PLQSVC(nq=4, gamma=0.2)
    print(cv_eval(X, y, "PennyLane QSVC (MI-4q)", "SMOTE+Jitter", splits=5, repeats=1))